In [1]:
import numpy as np
print(np.__version__)

1.26.4


In [2]:
import transformers, torch, accelerate
from transformers.utils import is_torch_available

print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(is_torch_available())

/Users/pardhu/Developer/TMF Classfier/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.38.2
2.2.2
0.27.2
True


In [3]:
cd ..

/Users/pardhu/Developer/TMF Classfier


/Users/pardhu/Developer/TMF Classfier/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
%pwd

'/Users/pardhu/Developer/TMF Classfier'

In [5]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [6]:
df = pd.read_csv("bert_3class_chunk_dataset.csv")

df.head()

,file_name,class,chunk_id,chunk_text,chunk_chars
0,Prot_009.pdf,protocol,54,"pd - l2 ). based on preclinical in vitro data,...",2024
1,Prot_007.pdf,protocol,60,regulatory requirements. clinical supplies sou...,1696
2,Prot_001.pdf,protocol,18,figures figure 1 study scheme....................,1758
3,Prot_005.pdf,protocol,51,"are required by the study protocol, including ...",2173
4,Prot_007.pdf,protocol,84,any aes that have an underlying true incidence...,2110


In [7]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["class"])

label2id = {
    label: int(idx)
    for idx, label in enumerate(label_encoder.classes_)
}

id2label = {
    int(idx): label
    for idx, label in enumerate(label_encoder.classes_)
}

label2id, id2label

({'protocol': 0, 'safety_report': 1, 'statistical_analysis_plan': 2},
 {0: 'protocol', 1: 'safety_report', 2: 'statistical_analysis_plan'})

In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_df.shape, test_df.shape

((1680, 6), (420, 6))

In [9]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/Users/pardhu/Developer/TMF Classfier/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
train_dataset = Dataset.from_pandas(
    train_df[["chunk_text", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["chunk_text", "label"]]
)

In [11]:
def tokenize_function(batch):
    return tokenizer(
        batch["chunk_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["chunk_text"])
test_dataset = test_dataset.remove_columns(["chunk_text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map: 100%|██████████| 420/420 [00:00<00:00, 1187.18 examples/s]


In [12]:
from transformers import AutoModelForSequenceClassification

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

/Users/pardhu/Developer/TMF Classfier/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [15]:
training_args = TrainingArguments(
    output_dir="bioclinicalbert_tmf_classifier",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=1,

    weight_decay=0.01,

    logging_dir="logs",
    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro"
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

 12%|█▏        | 50/420 [02:36<18:51,  3.06s/it]

{'loss': 1.0909, 'grad_norm': 7.560173511505127, 'learning_rate': 1.761904761904762e-05, 'epoch': 0.12}


 24%|██▍       | 100/420 [05:15<17:36,  3.30s/it]

{'loss': 0.8507, 'grad_norm': 12.75289249420166, 'learning_rate': 1.523809523809524e-05, 'epoch': 0.24}


 36%|███▌      | 150/420 [08:26<20:24,  4.53s/it]

...

                                                 
100%|██████████| 420/420 [37:45<00:00,  6.94s/it]

{'eval_loss': 0.4822213053703308, 'eval_accuracy': 0.780952380952381, 'eval_f1_macro': 0.7778636622303218, 'eval_runtime': 130.9476, 'eval_samples_per_second': 3.207, 'eval_steps_per_second': 0.802, 'epoch': 1.0}


100%|██████████| 420/420 [38:01<00:00,  5.43s/it]

{'train_runtime': 2281.4221, 'train_samples_per_second': 0.736, 'train_steps_per_second': 0.184, 'train_loss': 0.6857526915413993, 'epoch': 1.0}


TrainOutput(global_step=420, training_loss=0.6857526915413993, metrics={'train_runtime': 2281.4221, 'train_samples_per_second': 0.736, 'train_steps_per_second': 0.184, 'train_loss': 0.6857526915413993, 'epoch': 1.0})

In [18]:
results = trainer.evaluate()

results

100%|██████████| 105/105 [01:45<00:00,  1.00s/it]


{'eval_loss': 0.4822213053703308,
 'eval_accuracy': 0.780952380952381,
 'eval_f1_macro': 0.7778636622303218,
 'eval_runtime': 106.3009,
 'eval_samples_per_second': 3.951,
 'eval_steps_per_second': 0.988,
 'epoch': 1.0}

In [19]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

100%|██████████| 105/105 [02:03<00:00,  1.18s/it]


In [20]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)

                           precision    recall  f1-score   support

                 protocol       0.75      0.56      0.64       140
            safety_report       0.97      0.93      0.95       140
statistical_analysis_plan       0.66      0.86      0.75       140

                 accuracy                           0.78       420
                macro avg       0.79      0.78      0.78       420
             weighted avg       0.79      0.78      0.78       420



In [21]:
confusion_matrix(
    y_true,
    y_pred
)

array([[ 78,   3,  59],
       [  7, 130,   3],
       [ 19,   1, 120]])

In [22]:
trainer.save_model("saved_bioclinicalbert_tmf_3class")
tokenizer.save_pretrained("saved_bioclinicalbert_tmf_3class")

print("Model saved successfully.")

Model saved successfully.
